In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('processed_diabetes_data.csv')

X = df.drop(columns=['readmitted_binary'])
y = df['readmitted_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target distribution:\n", y_train.value_counts(normalize=True))

Train shape: (81412, 115)
Test shape: (20354, 115)
Train target distribution:
 readmitted_binary
0    0.888395
1    0.111605
Name: proportion, dtype: float64


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Scaling (Logistic Regression ke liye zaroori hai)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline model - class_weight='balanced' use karenge imbalance handle karne ke liye
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

ROC-AUC: 0.6504057212774415

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.67      0.78     18083
           1       0.17      0.54      0.26      2271

    accuracy                           0.66     20354
   macro avg       0.55      0.61      0.52     20354
weighted avg       0.84      0.66      0.72     20354


Confusion Matrix:
 [[12204  5879]
 [ 1044  1227]]


In [3]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', 
                              random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)  # RF ko scaling ki zaroorat nahi
rf_proba = rf.predict_proba(X_test)[:, 1]
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_proba))

# XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                     scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
xgb_proba = xgb.predict_proba(X_test)[:, 1]
print("XGBoost ROC-AUC:", roc_auc_score(y_test, xgb_proba))

Random Forest ROC-AUC: 0.6643852933826124


ValueError: feature_names must be string, and may not contain [, ] or <

In [4]:
import re

# Column names se special characters hatao
X_train.columns = [re.sub(r'[\[\]<>]', '', col) for col in X_train.columns]
X_test.columns = [re.sub(r'[\[\]<>]', '', col) for col in X_test.columns]

# Ab XGBoost dobara try karo
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                     scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
xgb_proba = xgb.predict_proba(X_test)[:, 1]
print("XGBoost ROC-AUC:", roc_auc_score(y_test, xgb_proba))

XGBoost ROC-AUC: 0.6769381670842942


In [5]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

xgb_base = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss')

random_search = RandomizedSearchCV(
    xgb_base, param_grid, n_iter=15, scoring='roc_auc', 
    cv=3, random_state=42, n_jobs=-1, verbose=1
)
random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best CV ROC-AUC:", random_search.best_score_)

# Test set pe evaluate karo
best_xgb = random_search.best_estimator_
best_proba = best_xgb.predict_proba(X_test)[:, 1]
print("Test ROC-AUC:", roc_auc_score(y_test, best_proba))

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Best params: {'subsample': 0.7, 'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
Best CV ROC-AUC: 0.6711438571565383
Test ROC-AUC: 0.6841657747594858


In [6]:
import joblib

joblib.dump(best_xgb, 'healthcare_readmission_model.pkl')
joblib.dump(scaler, 'healthcare_scaler.pkl')  # agar scaling use ki thi kahin

print("Model saved successfully")

Model saved successfully


In [7]:
import pandas as pd

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_xgb.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(15))

                                          feature  importance
9                                number_inpatient    0.162057
1                        discharge_disposition_id    0.053036
8                                number_emergency    0.027312
93                     diag_1_cat_Musculoskeletal    0.019553
10                               number_diagnoses    0.019306
87                                diabetesMed_Yes    0.016069
3                                time_in_hospital    0.015502
23                                     age_50-60)    0.014671
103                          diag_2_cat_Neoplasms    0.013527
96                         diag_1_cat_Respiratory    0.013358
38                                   metformin_No    0.011966
33   medical_specialty_Orthopedics-Reconstructive    0.011851
76                                     insulin_No    0.011594
110                            diag_3_cat_Missing    0.011477
16                                   race_Unknown    0.011468


In [8]:
joblib.dump(list(X_train.columns), 'model_columns.pkl')
print("Columns saved")

Columns saved


In [9]:
import requests

# Sample data - kisi ek row se values lo apne X_test se
sample = X_test.iloc[0].to_dict()

response = requests.post("http://127.0.0.1:5000/predict", json=sample)
print(response.json())

{'probability': 0.5196, 'readmission_risk': 'High Risk'}
